# Step 2 · Prepare supervised data

Versioned datasets: inspect, check quality and leakage, load a pack, edit an example.

*Same as the dashboard page, done in code.*

In [ ]:
# ── Setup (run first) ────────────────────────────────────────────────────────────
# Uses the same lab registry as the dashboard. To experiment in a sandbox instead:
#   os.environ["LAB_HOME"] = "/tmp/my_lab"   (before importing lab)
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, ROOT); os.chdir(ROOT)
os.environ.setdefault("HF_HUB_OFFLINE", "1")          # models are already cached locally
import pandas as pd
from lab import data, registry, paths, rewards
data.bootstrap_samples()
BASE = registry.ensure_base(paths.DEFAULT_BASE_MODEL)
if not registry.rewards():
    registry.add_reward(rewards.DEFAULT_RULES, note="Default draft rules")

def latest(kind):
    """Newest model id of a kind: base | sft | rl | rlhf | rm (or None)."""
    ms = [m for m in registry.models() if m["kind"] == kind]
    return ms[-1]["id"] if ms else None

def latest_ds(role):
    d = registry.latest_dataset(role)
    return d["id"] if d else None

print("Lab home:", paths.home(), "| base model:", BASE)

In [ ]:
SAVE = False   # set True to write new versions into your lab (they appear in the dashboard)

In [ ]:
pd.DataFrame(registry.datasets())[["id", "role", "n_rows", "source", "note"]]

In [ ]:
rows = data.load(latest_ds("sft_train"))
print(len(rows), "examples"); pd.DataFrame(rows).head()

In [ ]:
# Quality and leakage (a question in train AND validation/test makes scores look better than they are)
print(data.quality_warnings("sft_train", rows))
splits = {latest_ds(r): data.load(latest_ds(r)) for r in data.ROLES if latest_ds(r)}
print("leakage:", data.leakage(splits) or "none")

In [ ]:
# The 10 bundled supervised packs
pd.DataFrame(data.list_packs())[["slug", "title", "n_train", "n_val"]]

In [ ]:
# Edit one example → a new immutable version (old versions are kept)
new_rows = [dict(r) for r in rows]
new_rows[0]["completion"] = new_rows[0]["completion"].rstrip(".") + ", and tell the customer it is an estimate."
added, removed, changed = data.diff(rows, new_rows)
print(f"{len(changed)} changed")
if SAVE:
    print("saved", data.save_version("sft_train", new_rows, parent_id=latest_ds("sft_train"), note="notebook edit"))
    # data.load_pack("05_bank_fraud", mode="replace")   # load a pack as new sft_train + sft_val versions